In [1]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from pathlib import Path
from omegaconf import OmegaConf
from sklearn.decomposition import PCA
from sklearn.manifold import MDS

from toy_disentanglement.task import create_embedding_autoencoder, LatentClassificationDataset
from toy_disentanglement.metrics import classification_generalization_accuracy, regression_generalization_r2

## Load all corresponding runs

In [24]:
def flatten_dict(d, sep='.'):
    """Recursively flattens a nested dictionary."""
    items = []
    for k, v in d.items():
        new_key = k
        if isinstance(v, dict):
            items.extend(flatten_dict(v, sep).items())
        else:
            items.append((new_key, v))
    return dict(items)


def get_sweep_runs(sweep_id: int, run_dir: Path = Path("../runs/")):
    run_info = []
    for run in run_dir.iterdir():
        if not run.is_dir():
            continue
        if not (run / "config.yaml").exists():
            continue
        config = OmegaConf.load(run / "config.yaml")
        if config.get("sweep_id", None) == sweep_id:
            flat_config = flatten_dict(OmegaConf.to_container(config))
            run_info.append(
                {
                    **{k: v for k, v in flat_config.items() if ('$' not in str(v))},
                    "run_name": run.name,
                }
            )
    return pd.DataFrame(run_info)

In [26]:
run_df = get_sweep_runs(sweep_id=1)

## Show 